# LangChain Structured Outputs Guide

This notebook demonstrates how to extract structured information from Large Language Models (LLMs) using LangChain. We explore different schema definition approaches:
1. **Pydantic Models**
2. **TypedDict**
3. **Python Dataclasses**

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
os.environ["OPENAI_BASE_URL"] = os.getenv('OPENAI_BASE_URL')
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')


## Initialize the Chat Model

We instantiate our language model using `ChatGroq`. Below, we set up our primary LLM.

In [ ]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "qwen/qwen3.8-27b")

### 1. Using Pydantic

`Pydantic` is one of the most popular validation libraries in Python. Here, we define a flat Pydantic schema for `Movie` and use `with_structured_output` to force the model to conform to this structure.

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description= "The year the movie was released")
    director:str = Field(description= "The director of the movie")
    lead:str = Field(description="The lead actor of the movie")
    rating:float = Field(description= "The movie rating out of 10")

In [ ]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, client=<groq.resources.chat.completions.Completions object at 0x000002C1F914AC10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002C1F92A24D0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'lead': {'description': 'The lead actor of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'lead', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'metho

In [ ]:
response_without_structure = model.invoke("Provide the details of the movie The Matrix")
print(response_without_structure.content)

Here are the details for the movie **The Matrix**:

### **Basic Information**
- **Title**: The Matrix
- **Release Date**: March 31, 1999 (United States)
- **Runtime**: 136 minutes
- **Genre**: Action, Science Fiction, Thriller
- **Director**: The Wachowskis (Lana Wachowski and Lilly Wachowski)
- **Screenplay**: The Wachowskis
- **Producer**: Joel Silver
- **Music**: Don Davis
- **Cinematography**: Bill Pope
- **Starring**: Keanu Reeves, Laurence Fishburne, Carrie-Anne Moss, Hugo Weaving, Anthony Quinn

### **Plot Summary**
In the late 21st century, Thomas A. Anderson (Keanu Reeves), a day-time computer programmer and night-time hacker known as "Neo," discovers that his reality is an illusion. He is contacted by Morpheus (Laurence Fishburne) and Trinity (Carrie-Anne Moss), who reveal that humanity has been enslaved by sentient machines. The "Matrix" is a virtual reality system created by the machines to pacify human minds while using their bodies as energy sources.

Neo is offered the c

#### Parsing with Pydantic
Now we test the structured model. Notice the output conforms directly to our `Movie` model.

In [ ]:
response_with_structure = model_with_structure.invoke("Provide the details of the movie The Matrix")
print(response_with_structure)

title='The Matrix' year=1999 director='Lana Wachowski' lead='Keanu Reeves' rating=8.7


#### Message output

#### Retaining Raw LLM Response
By setting `include_raw=True`, we can inspect both the raw message metadata (tokens, finish reasons, etc.) and the parsed Pydantic object.

In [ ]:
class Movie(BaseModel):
    title:str = Field(...,description="The title of the movie")
    year:int = Field(...,description= "The year the movie was released")
    director:str = Field(...,description= "The director of the movie")
    lead:str = Field(..., description="The lead actor of the movie")
    rating:float = Field(...,description= "The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response_with_structure = model_with_structure.invoke("Provide the details of the movie The Matrix")
print(response_with_structure)

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'b78fbppd4', 'function': {'arguments': '{"director":"Lana Wachowski","lead":"Keanu Reeves","rating":8.7,"title":"The Matrix","year":1999}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 373, 'total_tokens': 458, 'completion_time': 0.232883363, 'completion_tokens_details': None, 'prompt_time': 0.02727185, 'prompt_tokens_details': None, 'queue_time': 0.064279337, 'total_time': 0.260155213}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a10725-5a27-7342-b3a9-ee062baf7fc1-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Lana Wachowski', 'lead': 'Keanu Reeves', 'rating': 8.7, 'title': 'The Matrix', 'year': 1999}, 'id': 'b78fbppd4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

### 2. Nested Pydantic Structure
We can represent more complex relationships by nesting models, such as listing multiple `Actor` objects inside the main `MovieDetails` response.

In [ ]:
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_str = model.with_structured_output(MovieDetails)

response_with_str = model_with_str.invoke("Provide the details of the movie The Matrix")
response_with_str

MovieDetails(title='The Matrix', year=1999, cast=[Actor(name='Keanu Reeves', role='Neo'), Actor(name='Laurence Fishburne', role='Morpheus'), Actor(name='Carrie-Anne Moss', role='Trinity'), Actor(name='Hugo Weaving', role='Agent Smith'), Actor(name='Joe Pantoliano', role='Cypher')], genres=['Action', 'Science Fiction'], budget=63.0)

## Using TypedDict

If you prefer standard Python dictionaries over Pydantic objects, you can use `TypedDict` along with `Annotated` to provide descriptions for each key.

In [ ]:
from typing_extensions import TypedDict, Annotated

In [ ]:
class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    lead: Annotated[str, ..., "The lead actor of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

#### Invoke Model with TypedDict
Below we invoke the model using our `MovieDict` definition.

In [ ]:
model_with_typedict = model.with_structured_output(MovieDict)
response_with_typedict = model_with_typedict.invoke("Provide the details of the movie The Batman(2022)")
response_with_typedict

{'director': 'Matt Reeves',
 'lead': 'Robert Pattinson',
 'rating': 7.8,
 'title': 'The Batman',
 'year': 2022}

### Nested TypedDict
Just like with Pydantic, you can nest definitions inside `TypedDict` for richer hierarchical payloads.

In [ ]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_tdstr = model.with_structured_output(MovieDetails)

response_with_tdstr = model_with_tdstr.invoke("Provide the details of the movie The Batman(2022)")
response_with_tdstr

{'budget': 185000000,
 'cast': [{'name': 'Robert Pattinson', 'role': 'Bruce Wayne / Batman'},
  {'name': 'Zoë Kravitz', 'role': 'Selina Kyle / Catwoman'},
  {'name': 'Paul Dano', 'role': 'The Riddler'},
  {'name': 'Colin Farrell', 'role': 'Oswald Cobblepot / The Penguin'},
  {'name': 'Jeffrey Wright', 'role': 'James Gordon'},
  {'name': 'Andy Serkis', 'role': 'Alfred Pennyworth'},
  {'name': 'Peter Sarsgaard', 'role': 'Carmine Falcone'},
  {'name': 'John Turturro', 'role': 'Victor Zsasz'}],
 'genres': ['Action', 'Crime', 'Drama', 'Mystery'],
 'title': 'The Batman',
 'year': 2022}

In [ ]:
print(model.profile)

None


## Using Dataclasses

Standard Python `@dataclass` objects can also serve as the schema blueprint for parsing or building custom agents.

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class Contactinfo:
    """Contact information for a person"""
    name: str #name of the person
    email: str #email of the person
    phone: str #phone number of the person

agent = create_agent(
    model = "gpt-5",
    response_format=Contactinfo
)

result = agent.invoke(
    {
        "messages":[{"role":"user", "content":"Extact the contact info from Shiva Ulavi, shivachetanulavi@gmail.com, (781)-301-1849" }]
    }
)

#### Extracting and Inspecting Dataclass Output
Finally, we query our agent to parse raw contact information into our structured dataclass.

In [ ]:
result

{'messages': [HumanMessage(content='Extact the contact info from Shiva Ulavi, shivachetanulavi@gmail.com, (781)-301-1849', additional_kwargs={}, response_metadata={}, id='92d5cbb7-2a56-4906-94cf-e365d569127b'),
  AIMessage(content='{"name":"Shiva Ulavi","email":"shivachetanulavi@gmail.com","phone":"(781)-301-1849"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 809, 'prompt_tokens': 184, 'total_tokens': 993, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'cost': 0.952334368128, 'cost_details': {'completion_cost_inr': 0.9260078170860001, 'prompt_cost_inr': 0.026326551042000004}, 'latency_ms': 9199}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerpr

In [ ]:
result['structured_response']

Contactinfo(name='Shiva Ulavi', email='shivachetanulavi@gmail.com', phone='(781)-301-1849')